In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetData(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(len(ModelData.timeStrings)), position=2, leave=False):
        for varName in varNames:
            dataDictionary[varName][t] = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], varName)[pIndices]

    return dataDictionary

def GetVarNames():
    varNames = ['z','LFC','W','QCQI','QV','THETA_v','THETA_e','BUOYANCY2'] #... ['HMC']
    return varNames

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def LimitTrackedArraysRows(trackedArrays,
                           limit=10000): #* use jobarray in the future
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            array = trackedArrays[parcelType][parcelDepth]
            trackedArrays[parcelType][parcelDepth]\
            = trackedArrays[parcelType][parcelDepth][:limit, :]

def ComputeTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(variableData.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

In [ ]:
def RunCode(trackedArrays):

    parcelTypes=['CL','nonCL']
    # parcelDepths=['SHALLOW','DEEP']
    parcelDepths=['SHALLOW'] #*testing
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes, position=0, leave=True):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", position=1, leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int); t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData(trackedArray)
            
            #getting data
            varNames=GetVarNames()
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = ComputeTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray

    return trajectoriesArrayDictionary

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    # trackedArray = MakeTestArray() #*testing
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    trajectoriesArrayDictionary = RunCode(trackedArrays)
    # TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
    #                                               trajectoriesArrayDictionary, dataName, t='all_times')

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
def PrepareData(dataDictionary,p=0,varName='W'):
    #getting
    z = dataDictionary['z'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; time_relative = time-time[0]
    z=z[mask]; LFC=LFC[mask]; z_relative = z-LFC
    data=data[mask]
    
    timeRelativeToWhenLFC=False
    if timeRelativeToWhenLFC:
        whenLFC=np.where(z_relative>0)[0][0]
        time_relative-=time_relative[whenLFC] #*time relative to time from LFC

    return time_relative,z_relative,data

# #testing
# p=3
# [time_relative,z_relative,data] = PrepareData(dataDictionary,p,varName='W')
# plt.plot(time_relative,z_relative)
# plt.ylabel('z - z_LFC (km)'); plt.xlabel('time since ascent (mins)')
# plt.axhline(0, color='grey', zorder=-10, linestyle='--',label='LFC')

def BinData(dataDictionary,varName='W'):    
    counts_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    weights_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    
    for p in range(dataDictionary['z'].shape[1]):
        time_relative, z_relative, data = PrepareData(dataDictionary, p, varName=varName)
        
        c, xedges, yedges = np.histogram2d(time_relative, z_relative, bins=[time_bins, z_bins])
        w, _, _ = np.histogram2d(time_relative, z_relative, bins=[xedges, yedges], weights=data)
        counts_all += c
        weights_all += w
    
    mean_data = np.where(counts_all > 0, weights_all / counts_all, np.nan)
    return mean_data
time_bins = np.arange(0, 60, 1)
z_bins = np.arange(-3000, 3000, 50)

def InterpolateBinnedData(mean_data):
    from scipy.interpolate import interp1d        
    # interpolate over empty time bins for each z level
    mean_data_filled = mean_data.copy()
    for iz in range(mean_data.shape[1]):
        row = mean_data[:, iz]
        valid = np.isfinite(row)
        if valid.sum() > 1:
            f = interp1d(time_bins[:-1][valid], row[valid], bounds_error=False, fill_value=np.nan)
            mean_data_filled[:, iz] = f(time_bins[:-1])
    return mean_data_filled
        
def PlotBinnedData(mean_data, interpolate=True, cmap='RdBu', symmetric=True, label='',
                   fig=None,axis=None, vmin_threshold=None):
    if axis is None:
            fig, axis = plt.subplots(figsize=(12, 5))
    if interpolate:
        mean_data = InterpolateBinnedData(mean_data)
    if vmin_threshold is not None:
        mean_data = np.where(mean_data >= vmin_threshold, mean_data, np.nan)
        
    if symmetric:
        extend = 'both'
        vmax = np.nanpercentile(np.abs(mean_data), 95)
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    else:
        extend = 'max'
        if vmin_threshold is not None:
            vmin = vmin_threshold 
        else:
            vmin = np.nanpercentile(mean_data, 5)
        vmax = np.nanpercentile(mean_data, 95)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        
    im = axis.pcolormesh(time_bins[:-1], z_bins[:-1], mean_data.T, cmap=cmap, norm=norm)
    plt.colorbar(im, ax=axis, label=label, extend=extend)
    axis.axhline(0, color='grey', linestyle='--', label='LFC')
    axis.set_xlabel('time since ascent (mins)')
    axis.set_ylabel('z - z_LFC (m)')
    axis.legend()

In [ ]:
##############################################
#PLOTTING

In [ ]:
varName='QV'
parcelTypes = ['CL','nonCL']

fig, axes = plt.subplots(2,1,figsize=(8, 10))
for axis,parcelType in zip(axes,parcelTypes):
    
    dataDictionary=trajectoriesArrayDictionary[parcelType]['SHALLOW']
    mean_data = BinData(dataDictionary, varName=varName)
    PlotBinnedData(mean_data, cmap='turbo', symmetric=False, label=varName, fig=fig,axis=axis)

In [ ]:
varName='THETA_v'
parcelTypes = ['CL','nonCL']

fig, axes = plt.subplots(2,1,figsize=(8, 10))
for axis,parcelType in zip(axes,parcelTypes):
    
    dataDictionary=trajectoriesArrayDictionary[parcelType]['SHALLOW']
    mean_data = BinData(dataDictionary, varName=varName)
    PlotBinnedData(mean_data, cmap='turbo', symmetric=False, label=varName, fig=fig,axis=axis)

In [ ]:
varName='W'
parcelTypes = ['CL','nonCL']

fig, axes = plt.subplots(2,1,figsize=(8, 10))
for axis,parcelType in zip(axes,parcelTypes):
    
    dataDictionary=trajectoriesArrayDictionary[parcelType]['SHALLOW']
    mean_data = BinData(dataDictionary, varName=varName)
    PlotBinnedData(mean_data, cmap='RdBu_r', symmetric=True, label=varName, fig=fig,axis=axis)

In [ ]:
varName='BUOYANCY2'
parcelTypes = ['CL','nonCL']

fig, axes = plt.subplots(2,1,figsize=(8, 10))
for axis,parcelType in zip(axes,parcelTypes):
    
    dataDictionary=trajectoriesArrayDictionary[parcelType]['SHALLOW']
    mean_data = BinData(dataDictionary, varName=varName)
    PlotBinnedData(mean_data, cmap='RdBu_r', symmetric=True, label=varName, fig=fig,axis=axis)

In [ ]:
varName='QCQI'
parcelTypes = ['CL','nonCL']

fig, axes = plt.subplots(2,1,figsize=(8, 10))
for axis,parcelType in zip(axes,parcelTypes):
    
    dataDictionary=trajectoriesArrayDictionary[parcelType]['SHALLOW']
    mean_data = BinData(dataDictionary, varName=varName)
    PlotBinnedData(mean_data, cmap='turbo', symmetric=False, label=varName, fig=fig,axis=axis, vmin_threshold=1e-6)

In [ ]:
########################################
#TESTING

In [ ]:
z=dataDictionary['z']
LFC=dataDictionary['LFC']
qcqi=dataDictionary['QCQI']

p=6
a=qcqi[:,p]
b=z[:,p]
c=LFC[:,p]

fig,axis=plt.subplots()
axis.plot(a,b)
axis.axhline(c[(b>c).argmax()],label='lfc',color='orange',linestyle='-',zorder=-5)
axis.axvline(1e-6,color='gray',linestyle='--',zorder=-10)
axis.set_xlabel('qc+qi (kg/kg)'); axis.set_ylabel('z (m)')